# NLP Assignment 2 — Project 7: Kaggle Execution Pipeline

**Student Roll:** 23CS30073  
**Task:** LLM Hallucination Detection on HaluEval QA  
**Environment:** Kaggle Notebook (GPU T4 x2 or P100)

> ⚠️ **IMPORTANT KAGGLE SETTINGS (Right Sidebar):**
> 1. **Settings → Accelerator**: Select **GPU T4 x2** (or GPU P100).
> 2. **Settings → Internet**: Toggle to **Internet ON** (Required to clone GitHub repo and download Hugging Face models).

In [ ]:
# Cell 1: Check GPU hardware & PyTorch device status
!nvidia-smi

import torch
print("\n" + "=" * 50)
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Device Count   : {torch.cuda.device_count()}")
print("=" * 50)

In [ ]:
# Cell 2: Setup Kaggle Working Directory & Clone Project Repository
import os
import shutil

# 1. Permanent working directory in Kaggle
KAGGLE_DIR = "/kaggle/working/NLP_Assignment_2"
os.makedirs(KAGGLE_DIR, exist_ok=True)

# 2. Enter the project folder
%cd /kaggle/working/NLP_Assignment_2

# 3. Clone or pull repo files directly into Kaggle working dir
if not os.path.exists("task1_colab.py"):
    !git clone https://github.com/ThilokT/NLP_Assignment_2.git .
else:
    !git pull origin main 2>/dev/null || true

print("\n✓ Working directory initialized:")
!pwd
!ls -la

In [ ]:
# Cell 3: Install Required Dependencies
!pip install -q transformers torch bert-score huggingface_hub accelerate pandas scikit-learn matplotlib joblib tqdm

In [ ]:
# Cell 4: Hugging Face Authentication
import os
from huggingface_hub import login

# Authenticate for google/gemma-2-2b-it
# Retrieve token from environment, Kaggle secrets, or user prompt
token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not token:
    from getpass import getpass
    token = getpass("Enter your Hugging Face token: ")

os.environ["HF_TOKEN"] = token
login(token=token)
print("✓ Hugging Face authenticated successfully!")

---
## Task 1: Knowledge-Conditioned Token Probability & Entropy Detection (~1 min on T4 GPU)
*Checkpoints and plots save directly into `/kaggle/working/NLP_Assignment_2`.*

In [ ]:
# Cell 5: Run Task 1
%cd /kaggle/working/NLP_Assignment_2
!python -u task1_colab.py

---
## Task 2: SelfCheckGPT (Resampling + BERTScore + DeBERTa NLI) (~18 mins on T4 GPU)
*Resamples and consistency scores save into checkpoints folder every 25 questions.*

In [ ]:
# Cell 6: Run Task 2
%cd /kaggle/working/NLP_Assignment_2
!python -u task2_colab.py

---
## Task 3: Stacked Ensemble Meta-Classifiers & Bootstrap Evaluation (~1 min)
*Trains Logistic Regression, Random Forest, Gradient Boosting with 5-fold CV and 2000 bootstrap CIs.*

In [ ]:
# Cell 7: Run Task 3
%cd /kaggle/working/NLP_Assignment_2
!python -u task3_colab.py

---
## Results Display & Inline Plots

In [ ]:
# Cell 8: Display Comparison Table & PR Curves Inline
%cd /kaggle/working/NLP_Assignment_2
import os
import pandas as pd
from IPython.display import Image, display

csv_path = "outputs/final_comparison_v2.csv"
if os.path.exists(csv_path):
    print("=== FINAL COMPARISON TABLE (TEST SET AUC-PR WITH 95% BOOTSTRAP CI) ===")
    display(pd.read_csv(csv_path))

for plot_file in ["outputs/task3_v2_pr_curves.png", "outputs/task1_v2_plots.png", "outputs/task2_v2_plots.png"]:
    if os.path.exists(plot_file):
        print(f"\nPlot: {plot_file}")
        display(Image(filename=plot_file))

---
## Download Results (Checkpoints, Outputs & Models as ZIP)

In [ ]:
# Cell 9: Package all Checkpoints, Models, and Outputs into a single ZIP for 1-click download
%cd /kaggle/working/NLP_Assignment_2
import os
from IPython.display import FileLink, display

zip_path = "/kaggle/working/nlp_assignment_2_results.zip"
!zip -q -r {zip_path} checkpoints/ outputs/

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✓ Results archive created: {zip_path} ({size_mb:.2f} MB)")
    print("Click link below to download directly to your computer:")
    display(FileLink("nlp_assignment_2_results.zip"))

print("\n=== Verification of Generated Checkpoints ===")
!ls -lh checkpoints/

print("\n=== Verification of Generated Outputs ===")
!ls -lh outputs/